# 第 5 天 · 参数与计费

> 📓 **这是第 5 天 的 Notebook 版**，内容和[课程网页](https://pollux1314.github.io/agent-course/#/day/5)一样。
> 先运行第一格「运行环境准备」，再从上往下逐格 Shift+Enter。你改的代码自动存在这个浏览器里；想恢复原样，在左侧文件列表右键这个文件 → Delete，刷新后会重新出现原版。
>
> 浏览器里的 Python（Pyodide）的限制：`input()`、`subprocess`、本地文件和数据库要用本地 `uv run`；访问外部网站需要对方允许跨域（DeepSeek、通义、GitHub API、httpbin 都可以）。本地脚本里的 `asyncio.run(main())` 在这里已改成 `await main()`；输出里看到 `SystemExit: ...` 是脚本主动退出（比如没填 Key），不是环境坏了。

In [ ]:
# ⚙️ 运行环境准备：打开 Notebook 后先运行这一格（Shift+Enter）
%pip install -q openai python-dotenv tiktoken
import sys
sys.argv = ["main.py"]  # 课程脚本里读命令行参数的地方走默认值
try:  # 同步 httpx / openai 在浏览器里走同步 XHR（和课程网页的运行器一致）
    import httpx._transports.jsfetch as _jf
    _jf.can_run_sync = lambda: False
except Exception:
    pass
import os, js
from pyodide.http import pyfetch
os.environ['TIKTOKEN_CACHE_DIR'] = '/tmp/tiktoken'; os.makedirs('/tmp/tiktoken', exist_ok=True)
if not os.path.exists('/tmp/tiktoken/9b5ad71b2ce5302211f9c61530b329a4922fc6a4'):
    # cl100k_base 编码文件改从本站下载（免跨域、国内可访问）
    _r = await pyfetch(str(js.location.href).split('/lite/')[0] + '/pyodide/cl100k_base.tiktoken')
    with open('/tmp/tiktoken/9b5ad71b2ce5302211f9c61530b329a4922fc6a4', 'wb') as f:
        f.write(await _r.bytes())
print("✓ 环境就绪，可以往下运行了")

In [ ]:
# 🔑 这一课要调用大模型：把 Key 填在引号里再运行这一格（只存在这次打开的浏览器内存里，别分享这个 Notebook）
import os
LLM_API_KEY = ""
os.environ["LLM_BASE_URL"] = "https://api.deepseek.com"   # 通义：https://dashscope.aliyuncs.com/compatible-mode/v1
os.environ["LLM_MODEL"] = "deepseek-chat"                 # 通义：qwen-plus
os.environ.setdefault("PRICE_IN_PER_M", "2")    # 输入单价（元/百万 token），以服务商官网为准
os.environ.setdefault("PRICE_OUT_PER_M", "8")   # 输出单价
if LLM_API_KEY:
    os.environ["LLM_API_KEY"] = LLM_API_KEY
print("Key：", "已设置" if os.getenv("LLM_API_KEY") else "未设置（会走缺 Key 的报错分支）")

In [ ]:
import tiktoken

enc = tiktoken.get_encoding("cl100k_base")   # OpenAI 的分词器，对别家模型只是近似
for s in ["Hello world", "你好，世界", "public static void main(String[] args)", "ReentrantLock"]:
    ids = enc.encode(s)
    print(f"{s!r:45} -> {len(ids):2} tokens {[enc.decode([i]) for i in ids]}")

In [ ]:
# /// script
# requires-python = ">=3.11"
# dependencies = ["openai>=1.40", "python-dotenv>=1.0", "tiktoken>=0.7"]
# ///
"""第 5 天：参数与计费。运行：uv run day05_params.py "你的问题" """
import os
import sys
import time

import tiktoken
from dotenv import load_dotenv
from openai import APIConnectionError, APIStatusError, AuthenticationError, OpenAI

load_dotenv()
API_KEY = os.getenv("LLM_API_KEY")
BASE_URL = os.getenv("LLM_BASE_URL", "https://api.deepseek.com")
MODEL = os.getenv("LLM_MODEL", "deepseek-chat")
# 单价：元 / 百万 token。去厂商官网“价格”页查当前价格填进 .env，这里不写死
PRICE_IN = float(os.getenv("PRICE_IN_PER_M", "0"))
PRICE_OUT = float(os.getenv("PRICE_OUT_PER_M", "0"))

ENC = tiktoken.get_encoding("cl100k_base")  # OpenAI 的分词器，对别家模型只是近似


def count_tokens(text: str) -> int:
    return len(ENC.encode(text))


def cost(prompt_tokens: int, completion_tokens: int) -> float:
    return (prompt_tokens * PRICE_IN + completion_tokens * PRICE_OUT) / 1_000_000


def demo_tokens() -> None:
    """不需要 API Key：先直观感受一下 token 是什么"""
    for s in ["Hello world", "你好，世界", "public static void main(String[] args)", "ReentrantLock"]:
        ids = ENC.encode(s)
        pieces = [ENC.decode([i]) for i in ids]
        print(f"{s!r:45} -> {len(ids):2} tokens {pieces}")
    print()


def run(client: OpenAI, question: str, temperature: float, max_tokens: int) -> None:
    messages = [
        {"role": "system", "content": "你是一个有创意的文案，回答一句话。"},
        {"role": "user", "content": question},
    ]
    est = sum(count_tokens(m["content"]) for m in messages)
    start = time.perf_counter()
    resp = client.chat.completions.create(
        model=MODEL, messages=messages, temperature=temperature, max_tokens=max_tokens
    )
    ms = (time.perf_counter() - start) * 1000
    u = resp.usage
    c = resp.choices[0]
    print(f"| T={temperature:<3} | 预估输入≈{est:3} | 实际 {u.prompt_tokens:3}+{u.completion_tokens:3} "
          f"| ¥{cost(u.prompt_tokens, u.completion_tokens):.6f} | {ms:5.0f}ms | {c.finish_reason:6} "
          f"| {c.message.content.strip()[:60]!r}")


if __name__ == "__main__":
    demo_tokens()
    if not API_KEY:
        sys.exit("没找到 LLM_API_KEY：上面的 token 演示不需要 Key；要调模型请先配好 .env")
    client = OpenAI(api_key=API_KEY, base_url=BASE_URL, timeout=30)
    q = " ".join(sys.argv[1:]) or "给一家只卖咖啡的程序员主题小店起个名字"
    try:
        for t in (0.0, 0.7, 1.5):
            for _ in range(3):  # 同一个温度跑 3 次，看输出稳不稳定
                run(client, q, temperature=t, max_tokens=60)
        print("\n--- max_tokens=5，看看会发生什么 ---")
        run(client, q, temperature=0.7, max_tokens=5)
    except AuthenticationError:
        sys.exit("401：API Key 不对或已失效，检查 .env")
    except APIConnectionError as e:
        sys.exit(f"连不上 {BASE_URL}：{e}")
    except APIStatusError as e:
        sys.exit(f"HTTP {e.status_code}：{e.message}")

In [ ]:
PRICE_IN_PER_M = 2.0    # 示例数字（元 / 百万 token），改成你查到的单价
PRICE_OUT_PER_M = 8.0

req_per_day = 10_000
in_tokens, out_tokens = 2000, 300

per_req = (in_tokens * PRICE_IN_PER_M + out_tokens * PRICE_OUT_PER_M) / 1_000_000
print(f"单次 ¥{per_req:.5f}，每天 ¥{per_req * req_per_day:.2f}，每月 ¥{per_req * req_per_day * 30:.2f}")

# 多轮对话：每一轮都要把历史全部带上
history = 0
for turn in range(1, 11):
    prompt = 2000 + history
    history += 300 + 200          # 上一轮的回答 + 用户新问题，粗略估计
    if turn in (1, 5, 10):
        print(f"第 {turn:2} 轮输入 ≈ {prompt} tokens")